In [7]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
import joblib
import os

In [8]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [9]:
UNLABELED_PARQUET = "train_unlabelled.parquet"

table = pq.read_table("train_unlabelled.parquet", columns=[])
print(f"Number of rows: {table.num_rows:,}")

unlabelled_count = table.num_rows


Number of rows: 36,029,216


In [10]:
LABELLED_PARQUET = "train_labelled.parquet"

table = pq.read_table(LABELLED_PARQUET, columns=["party", "sentiment"])

labelled_count = table.num_rows

party_counts = {
    str(row["values"].as_py()): row["counts"].as_py()
    for row in pc.value_counts(table["party"])
}

sentiment_by_party = {}
parties = table["party"].unique().to_pylist()
for party in parties:
    mask = pc.equal(table["party"], party) 
    subset = table.filter(mask)

    counts = pc.value_counts(subset["sentiment"])
    sentiment_by_party[party] = {
        str(row["values"].as_py()): row["counts"].as_py()
        for row in counts
    }

print(f"\nLabelled samples: {labelled_count:,}")
print(f"Party distribution: {party_counts}")
for party, dist in sentiment_by_party.items():
    print(f"  {party}: {dist}")



Labelled samples: 864,378
Party distribution: {'republican': 450766, 'democrat': 413612}
  republican: {'positive': 256792, 'negative': 193974}
  democrat: {'positive': 159712, 'negative': 253900}


In [11]:
print(f"{(labelled_count/unlabelled_count*100):.2f}")

2.40


In [14]:
PREDICTIONS = "predictions.parquet"

table = pq.read_table(PREDICTIONS, columns=["party", "sentiment"])

count = table.num_rows

party_counts = {
    str(row["values"].as_py()): row["counts"].as_py()
    for row in pc.value_counts(table["party"])
}

sentiment_by_party = {}
parties = table["party"].unique().to_pylist()
for party in parties:
    mask = pc.equal(table["party"], party) 
    subset = table.filter(mask)

    counts = pc.value_counts(subset["sentiment"])
    sentiment_by_party[party] = {
        str(row["values"].as_py()): row["counts"].as_py()
        for row in counts
    }


print(f"Party distribution: {party_counts}")
for party, dist in sentiment_by_party.items():
    print(f"  {party}: {dist}")


Party distribution: {'democrat': 26500640, 'republican': 9528576}
  democrat: {'negative': 14948798, 'positive': 11551842}
  republican: {'negative': 4542921, 'positive': 4985655}


In [15]:
df = pd.read_parquet('predictions.parquet', engine='pyarrow')
df.head()

,id,clean_text,text,party,sentiment
0,1844361929781113331,many failure bidens administration directly traced harris deliverable failed win biden mentally deficient president pushed loses biden say told beat trump panicked one debate,"@a_newsman Many failures of Bidens administration are directly traced to Harris on deliverables she failed at. If she wins Biden is the mentally deficient president that had to be pushed out, if she loses Biden says “I told you so. I beat Trump once but you panicked after one debate.”",democrat,negative
1,1844361929550332320,lie biden told american misinformation set exactly 's warning elected suppress free speech,"@atensnut The lies Biden told Americans about ""misinformation"" is a set up to do exactly what she's warning she will do if elected - suppress free speech!",democrat,negative
2,1844361928925384711,someone talented fundraising team,@TheDemocrats here’s someone very talented for your fundraising team.,democrat,positive
3,1844361928585654516,chipper usd card working expressionless face,@LifeOfNapaul Chipper USD card isn’t working 😑,democrat,positive
4,1844361927495123076,yes n't done live interview real journalist since biden dropped n't even stated policy economy,@MTGrepp Yes. She hasn't done a live interview with a real journalist since Biden dropped out. She doesn't even a stated policy on the economy.,democrat,positive
